In [1]:
import json
from collections import Counter, defaultdict
import statistics

path = "../../../results/phase1_easy/repair/details.jsonl"

def load_results(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = load_results(path)
len(data)

def group_by_task(data):
    grouped = defaultdict(list)
    for row in data:
        grouped[row["task_id"]].append(row)

    for task_id in grouped:
        grouped[task_id] = sorted(grouped[task_id], key=lambda x: x["attempt_idx"])
    return grouped

grouped = group_by_task(data)

print(f"📂 File: {path}")
print(f"🧾 Total attempt rows: {len(data)}")
print(f"🧩 Total unique tasks: {len(grouped)}")

📂 File: ../../../results/phase1_easy/repair/details.jsonl
🧾 Total attempt rows: 245
🧩 Total unique tasks: 164


### task-level final summary
- 총 164문제 중 최종 몇 개 해결했는지
- initial fail이 final pass로 바뀐 개수

> Final Success Rate   
> Repair Gain

In [2]:
# task-level summary 함수
def summarize_task_level(grouped):
    total = len(grouped)

    final_pass = 0
    final_timeout = 0
    final_fail = 0

    solved_on_attempt = Counter()
    repair_gain = 0

    for task_id, attempts in grouped.items():
        first = attempts[0]
        last = attempts[-1]

        if last["status"] == "pass":
            final_pass += 1
        elif last["status"] == "timeout":
            final_timeout += 1
        else:
            final_fail += 1

        first_success_attempt = None
        for row in attempts:
            if row["status"] == "pass":
                first_success_attempt = row["attempt_idx"]
                break

        if first_success_attempt is not None:
            solved_on_attempt[first_success_attempt] += 1

        if first["status"] != "pass" and last["status"] == "pass":
            repair_gain += 1

    return {
        "total": total,
        "final_pass": final_pass,
        "final_fail": final_fail,
        "final_timeout": final_timeout,
        "final_success_rate": final_pass / total if total > 0 else 0.0,
        "solved_on_attempt": solved_on_attempt,
        "repair_gain": repair_gain,
    }

task_summary = summarize_task_level(grouped)

# task-level summary 출력
print("=" * 60)
print("📊 Task-level Final Summary")
print(f"Total tasks: {task_summary['total']}")
print(f"Final Pass: {task_summary['final_pass']}")
print(f"Final Fail: {task_summary['final_fail']}")
print(f"Final Timeout: {task_summary['final_timeout']}")
print(f"Final Success Rate: {task_summary['final_success_rate']:.4f}")
print(f"Repair Gain (initial fail -> final pass): {task_summary['repair_gain']}")

print("\n🎯 Solved on Attempt")
for attempt_idx, count in sorted(task_summary["solved_on_attempt"].items()):
    print(f"Attempt {attempt_idx}: {count}")

📊 Task-level Final Summary
Total tasks: 164
Final Pass: 138
Final Fail: 26
Final Timeout: 0
Final Success Rate: 0.8415
Repair Gain (initial fail -> final pass): 20

🎯 Solved on Attempt
Attempt 0: 118
Attempt 1: 11
Attempt 2: 9


In [3]:
# attempt-level summary 함수
def summarize_attempt_level(data):
    by_attempt = defaultdict(list)
    for row in data:
        by_attempt[row["attempt_idx"]].append(row)

    summary = {}

    for attempt_idx, rows in sorted(by_attempt.items()):
        total = len(rows)
        passed = sum(1 for r in rows if r["status"] == "pass")
        timeout = sum(1 for r in rows if r["status"] == "timeout")
        failed = total - passed - timeout

        error_counter = Counter(
            r["error_type"] for r in rows
            if r["status"] != "pass" and r["error_type"] is not None
        )

        latencies = [r["latency_sec"] for r in rows if r["latency_sec"] is not None]

        summary[attempt_idx] = {
            "total": total,
            "pass": passed,
            "fail": failed,
            "timeout": timeout,
            "pass_rate": passed / total if total > 0 else 0.0,
            "error_counter": error_counter,
            "avg_latency": statistics.mean(latencies) if latencies else None,
            "max_latency": max(latencies) if latencies else None,
        }

    return summary

attempt_summary = summarize_attempt_level(data)
print("=" * 60)
print("📈 Attempt-level Summary")

for attempt_idx, s in attempt_summary.items():
    print("\n" + "-" * 40)
    print(f"Attempt {attempt_idx}")
    print(f"Total: {s['total']}")
    print(f"Pass: {s['pass']}")
    print(f"Fail: {s['fail']}")
    print(f"Timeout: {s['timeout']}")
    print(f"Pass Rate: {s['pass_rate']:.4f}")

    if s["avg_latency"] is not None:
        print(f"Avg Latency: {s['avg_latency']:.3f}s")
        print(f"Max Latency: {s['max_latency']:.3f}s")

    print("Error Type Distribution:")
    if s["error_counter"]:
        for err, cnt in s["error_counter"].most_common():
            print(f"  {err}: {cnt}")
    else:
        print("  (none)")

📈 Attempt-level Summary

----------------------------------------
Attempt 0
Total: 164
Pass: 118
Fail: 46
Timeout: 0
Pass Rate: 0.7195
Avg Latency: 2.126s
Max Latency: 6.628s
Error Type Distribution:
  assertion_error: 28
  syntax_error: 13
  name_error: 3
  type_error: 1
  value_error: 1

----------------------------------------
Attempt 1
Total: 46
Pass: 11
Fail: 35
Timeout: 0
Pass Rate: 0.2391
Avg Latency: 2.799s
Max Latency: 6.698s
Error Type Distribution:
  syntax_error: 18
  assertion_error: 15
  name_error: 2

----------------------------------------
Attempt 2
Total: 35
Pass: 9
Fail: 26
Timeout: 0
Pass Rate: 0.2571
Avg Latency: 2.241s
Max Latency: 6.709s
Error Type Distribution:
  assertion_error: 11
  syntax_error: 11
  name_error: 4


In [4]:
# repaired case 샘플 보기
print("=" * 60)
print("🔧 Sample Repaired Cases (up to 3)")

shown = 0
for task_id, attempts in grouped.items():
    first = attempts[0]
    last = attempts[-1]

    if first["status"] != "pass" and last["status"] == "pass":
        print("-" * 40)
        print(f"Task: {task_id}")
        print(f"Initial Status: {first['status']} ({first['error_type']})")
        print(f"Final Status: {last['status']}")
        print(f"Solved at attempt: {last['attempt_idx']}")
        shown += 1

        if shown >= 3:
            break

if shown == 0:
    print("No repaired cases found.")

🔧 Sample Repaired Cases (up to 3)
----------------------------------------
Task: HumanEval/2
Initial Status: fail (assertion_error)
Final Status: pass
Solved at attempt: 2
----------------------------------------
Task: HumanEval/27
Initial Status: fail (syntax_error)
Final Status: pass
Solved at attempt: 1
----------------------------------------
Task: HumanEval/33
Initial Status: fail (assertion_error)
Final Status: pass
Solved at attempt: 1


In [5]:
# unrepaired case 샘플 보기
print("=" * 60)
print("❌ Sample Unrepaired Cases (up to 3)")

shown = 0
for task_id, attempts in grouped.items():
    first = attempts[0]
    last = attempts[-1]

    if last["status"] != "pass":
        print("-" * 40)
        print(f"Task: {task_id}")
        print(f"Initial Status: {first['status']} ({first['error_type']})")
        print(f"Final Status: {last['status']} ({last['error_type']})")
        print(f"Attempts used: {len(attempts)}")
        print(f"Final Error Msg: {last['error_message']}")
        shown += 1

        if shown >= 3:
            break

if shown == 0:
    print("All tasks were solved.")

❌ Sample Unrepaired Cases (up to 3)
----------------------------------------
Task: HumanEval/10
Initial Status: fail (assertion_error)
Final Status: fail (name_error)
Attempts used: 3
Final Error Msg: Traceback (most recent call last):
  File "/tmp/tmpxz0ub69l.py", line 20, in <module>
    s
NameError: name 's' is not defined

----------------------------------------
Task: HumanEval/26
Initial Status: fail (assertion_error)
Final Status: fail (assertion_error)
Attempts used: 3
Final Error Msg: Traceback (most recent call last):
  File "/tmp/tmp2ma1gk50.py", line 26, in <module>
    check(remove_duplicates)
  File "/tmp/tmp2ma1gk50.py", line 22, in check
    assert candidate([]) == []
           ^^^^^^^^^^^^^^^^^^^
AssertionError

----------------------------------------
Task: HumanEval/37
Initial Status: fail (assertion_error)
Final Status: fail (syntax_error)
Attempts used: 3
Final Error Msg:   File "/tmp/tmp84s17213.py", line 32
    [Execution Output]
     ^^^^^^^^^^^^^^^^
SyntaxErro

In [6]:
# repaired examples
repaired = []
for task_id, attempts in grouped.items():
    first = attempts[0]
    last = attempts[-1]
    if first["status"] != "pass" and last["status"] == "pass":
        repaired.append((task_id, attempts))

len(repaired)



20

In [7]:
# 예시 하나 보기
task_id, attempts = repaired[0]
print(f"TASK: {task_id}")

for row in attempts:
    print("=" * 80)
    print(f"Attempt {row['attempt_idx']} | status={row['status']} | error_type={row['error_type']}")
    print("\n--- RAW OUTPUT (head) ---")
    print(row["raw_output"][:500])
    print("\n--- GENERATED CODE (head) ---")
    print(row["generated_code"][:500])
    print("\n--- ERROR MESSAGE ---")
    print(row["error_message"])

TASK: HumanEval/2
Attempt 0 | status=fail | error_type=assertion_error

--- RAW OUTPUT (head) ---
    return number - int(number)

# Test cases to validate the correctness of the function
assert truncate_number(3.5) == 0.5
assert truncate_number(4.75) == 0.75
assert truncate_number(9.99) == 0.99
assert truncate_number(0.123) == 0.123
assert truncate_number(10.0) == 0.0
```

--- GENERATED CODE (head) ---


def truncate_number(number: float) -> float:
    """ Given a positive floating point number, it can be decomposed into
    and integer part (largest integer smaller than given number) and decimals
    (leftover part always smaller than 1).

    Return the decimal part of the number.
    >>> truncate_number(3.5)
    0.5
    """
    return number - int(number)

# Test cases to validate the correctness of the function
assert truncate_number(3.5) == 0.5
assert truncate_number(4.75) == 0.75
assert 

--- ERROR MESSAGE ---
Traceback (most recent call last):
  File "/tmp/tmptriq35vg.py", line

In [8]:
import pandas as pd

df = pd.DataFrame(data)
df.head()

,dataset,task_id,entry_point,method,model_name,attempt_idx,status,passed,timeout,raw_output,generated_code,error_type,error_message,latency_sec
0,humaneval,HumanEval/0,has_close_elements,repair,Qwen/Qwen2.5-Coder-7B-Instruct,0,pass,True,False,for i in range(len(numbers)):\n for...,from typing import List\n\n\ndef has_close_ele...,None,None,1.875275
1,humaneval,HumanEval/1,separate_paren_groups,repair,Qwen/Qwen2.5-Coder-7B-Instruct,0,pass,True,False,result = []\n current_group = ''\n d...,from typing import List\n\n\ndef separate_pare...,None,None,1.406838
2,humaneval,HumanEval/2,truncate_number,repair,Qwen/Qwen2.5-Coder-7B-Instruct,0,fail,False,False,return number - int(number)\n\n# Test case...,\n\ndef truncate_number(number: float) -> floa...,assertion_error,"Traceback (most recent call last):\n File ""/t...",1.266613
3,humaneval,HumanEval/2,truncate_number,repair,Qwen/Qwen2.5-Coder-7B-Instruct,1,fail,False,False,```python\ndef truncate_number(number: float) ...,\n\ndef truncate_number(number: float) -> floa...,syntax_error,"File ""/tmp/tmpi936wzw1.py"", line 24\n The...",1.867548
4,humaneval,HumanEval/2,truncate_number,repair,Qwen/Qwen2.5-Coder-7B-Instruct,2,pass,True,False,```python\ndef truncate_number(number: float) ...,\n\ndef truncate_number(number: float) -> floa...,None,None,1.215935


In [9]:
rows = []
for task_id, attempts in grouped.items():
    first = attempts[0]
    last = attempts[-1]

    solved_on = None
    for row in attempts:
        if row["status"] == "pass":
            solved_on = row["attempt_idx"]
            break

    rows.append({
        "task_id": task_id,
        "initial_status": first["status"],
        "initial_error_type": first["error_type"],
        "final_status": last["status"],
        "final_error_type": last["error_type"],
        "num_attempts": len(attempts),
        "solved_on_attempt": solved_on,
        "repaired": first["status"] != "pass" and last["status"] == "pass",
    })

task_df = pd.DataFrame(rows)
task_df.head()

,task_id,initial_status,initial_error_type,final_status,final_error_type,num_attempts,solved_on_attempt,repaired
0,HumanEval/0,pass,None,pass,None,1,0.0,False
1,HumanEval/1,pass,None,pass,None,1,0.0,False
2,HumanEval/2,fail,assertion_error,pass,None,3,2.0,True
3,HumanEval/3,pass,None,pass,None,1,0.0,False
4,HumanEval/4,pass,None,pass,None,1,0.0,False


In [10]:
task_df["repaired"].value_counts()

repaired
False    144
True      20
Name: count, dtype: int64

In [11]:
task_df.groupby(["initial_error_type", "repaired"]).size().sort_values(ascending=False)

initial_error_type  repaired
assertion_error     False       21
syntax_error        True        11
assertion_error     True         7
name_error          False        2
syntax_error        False        2
name_error          True         1
type_error          False        1
value_error         True         1
dtype: int64

In [12]:
task_df["solved_on_attempt"].value_counts(dropna=False).sort_index()

solved_on_attempt
0.0    118
1.0     11
2.0      9
NaN     26
Name: count, dtype: int64

---

In [13]:
repaired_cases = []
for task_id, attempts in grouped.items():
    first = attempts[0]
    last = attempts[-1]
    if first["status"] != "pass" and last["status"] == "pass":
        repaired_cases.append((task_id, attempts))

len(repaired_cases)

for task_id, attempts in repaired_cases[:3]:
    print("=" * 100)
    print(f"TASK: {task_id}")
    for row in attempts:
        print("-" * 60)
        print(
            f"attempt={row['attempt_idx']} "
            f"status={row['status']} "
            f"error_type={row['error_type']}"
        )
        print("\n[RAW OUTPUT HEAD]")
        print(row["raw_output"][:400])
        print("\n[GENERATED CODE HEAD]")
        print(row["generated_code"][:400])
        print("\n[ERROR MESSAGE]")
        print(row["error_message"])
        print()

TASK: HumanEval/2
------------------------------------------------------------
attempt=0 status=fail error_type=assertion_error

[RAW OUTPUT HEAD]
    return number - int(number)

# Test cases to validate the correctness of the function
assert truncate_number(3.5) == 0.5
assert truncate_number(4.75) == 0.75
assert truncate_number(9.99) == 0.99
assert truncate_number(0.123) == 0.123
assert truncate_number(10.0) == 0.0
```

[GENERATED CODE HEAD]


def truncate_number(number: float) -> float:
    """ Given a positive floating point number, it can be decomposed into
    and integer part (largest integer smaller than given number) and decimals
    (leftover part always smaller than 1).

    Return the decimal part of the number.
    >>> truncate_number(3.5)
    0.5
    """
    return number - int(number)

# Test cases to validate the correct

[ERROR MESSAGE]
Traceback (most recent call last):
  File "/tmp/tmptriq35vg.py", line 17, in <module>
    assert truncate_number(9.99) == 0.99
       

In [14]:
unrepaired_cases = []
for task_id, attempts in grouped.items():
    last = attempts[-1]
    if last["status"] != "pass":
        unrepaired_cases.append((task_id, attempts))

len(unrepaired_cases)

for task_id, attempts in unrepaired_cases[:3]:
    print("=" * 100)
    print(f"TASK: {task_id}")
    for row in attempts:
        print("-" * 60)
        print(
            f"attempt={row['attempt_idx']} "
            f"status={row['status']} "
            f"error_type={row['error_type']}"
        )
        print("\n[RAW OUTPUT HEAD]")
        print(row["raw_output"][:400])
        print("\n[GENERATED CODE HEAD]")
        print(row["generated_code"][:400])
        print("\n[ERROR MESSAGE]")
        print(row["error_message"])
        print()

TASK: HumanEval/10
------------------------------------------------------------
attempt=0 status=fail error_type=assertion_error

[RAW OUTPUT HEAD]
    for i in range(len(string), -1, -1):
        if is_palindrome(string[i:]):
            return string + string[:i][::-1]
    return string


if __name__ == '__main__':
    import doctest
    doctest.testmod()  # run tests in docstrings
```

[GENERATED CODE HEAD]


def is_palindrome(string: str) -> bool:
    """ Test if given string is a palindrome """
    return string == string[::-1]


def make_palindrome(string: str) -> str:
    """ Find the shortest palindrome that begins with a supplied string.
    Algorithm idea is simple:
    - Find the longest postfix of supplied string that is a palindrome.
    - Append to the end of the string reverse of a string

[ERROR MESSAGE]
Traceback (most recent call last):
  File "/tmp/tmp1e6zetfz.py", line 41, in <module>
    check(make_palindrome)
  File "/tmp/tmp1e6zetfz.py", line 36, in check
    ass

In [15]:
from collections import Counter

transition_counter = Counter()

for task_id, attempts in grouped.items():
    if len(attempts) >= 2:
        first_err = attempts[0]["error_type"]
        last_err = attempts[-1]["error_type"]
        transition_counter[(first_err, last_err)] += 1

transition_counter.most_common(20)

[(('assertion_error', 'assertion_error'), 11),
 (('syntax_error', None), 11),
 (('assertion_error', None), 7),
 (('assertion_error', 'syntax_error'), 7),
 (('assertion_error', 'name_error'), 3),
 (('syntax_error', 'syntax_error'), 2),
 (('name_error', 'name_error'), 1),
 (('type_error', 'syntax_error'), 1),
 (('value_error', None), 1),
 (('name_error', 'syntax_error'), 1),
 (('name_error', None), 1)]